# **Gene clustering project (acute myeloid leukemia and acute lymphoblastic leukemia)**

In [26]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 

About the dataset: 
* Gene expression measurements by DNA microarray from patients with AML or ALL
* AML and ALL samples from bone marrow and peripheral blood 
* Dataset from: Golub and al., Molecular Classification of Cancer: Class Discovery and Class Prediction by Gene Expression, Science 286:531-537. (1999). Published: 1999.10.14
* The authors used the dataset to classify the type of cancer in each patient by their gene expression
* 1 column = 1 patient ; 1 row = 1 gene

In [ ]:
# Importing the datasets

genes_train = pd.read_csv('../input/datasets/crawford/gene-expression/data_set_ALL_AML_train.csv')
genes_independent = pd.read_csv('../input/datasets/crawford/gene-expression/data_set_ALL_AML_independent.csv')
cancer_type = pd.read_csv('../input/datasets/crawford/gene-expression/actual.csv')

## **EDA**

Goals of this EDA: create a cleaned and scaled input for the clustering and classification models
* distribution
* NaN absence
* gene expression variance (to select the most informative genes)

In [ ]:
# Number of rows and columns

genes_train.shape # 7129 genes (rows) and 38 patients (columns)

In [ ]:
# Structure of the dataset

genes_train.head()

In [ ]:
# Columns in the dataset

genes_train.columns 

# the call column: whether or not a gene is present in the sample based on the measurement of its expression

In [ ]:
# Keeping only the columns with the expression measurement and getting rid of the "call" columns 

genes_train_ok = genes_train.copy()
cols_to_drop = [col for col in genes_train_ok.columns if 'call' in col.lower()]
print(cols_to_drop) # making sure that the right columns are about to be dropped from the dataset
genes_train_ok = genes_train_ok.drop(genes_train_ok[cols_to_drop], axis=1)
genes_train_ok.shape
genes_train_ok

In [ ]:
# Transposing the rows and the columns so that the genes become the features and the patients, the samples

genes_train_ok = genes_train_ok.transpose()
genes_train_ok.head()

In [ ]:
print(genes_train_ok.columns.tolist()) # all the genes

In [ ]:
# Preparing the data for distribution visualization

print(genes_train_ok.dtypes) # type: object 

# Putting genes information in another dataset
genes_info = genes_train_ok.iloc[0:2,:]
genes_train_ok = genes_train_ok.drop(['Gene Description', 'Gene Accession Number'], axis=0)

# Converting the numerical dataset to float
genes_train_ok = genes_train_ok.apply(pd.to_numeric, errors = 'coerce')

# Checking the NaN after the conversion
genes_train_ok.isna().mean().mean()

In [ ]:
# Checking that the gene information dataset has been successfully created

genes_info.head() 
genes_train_ok.index 

In [ ]:
# Global distribution of the dataset

plt.hist(genes_train_ok.values.flatten(), bins=100)

In [ ]:
sns.kdeplot(data = genes_train_ok.values.flatten())

The values are centered around 0 meaning the dataset is symetric/centered. 

In [ ]:
# Distribution per patient

genes_train_ok.T.hist(bins=50, figsize=(15,10))

All distributions are centered around 0. All patients are comparable in terms of scale and there is no biased patient (good for the clustering). 
However, it does not mean that the patients are the same, they just have the same gene expression patterns. 

Both global and per patient distributions are centered around 0, suggesting that the data has been normalized in advance. The uniformity observed between patients indicates that there is no major scale bias, which is good for the clustering. However, the distributions do not allow us to identify patient groups since the relevant differences rely on specific gene expression patterns. 

In [ ]:
# NaN

genes_train_ok.isna().sum()

# No NaN in the dataset 

In [ ]:
# Gene variances: identifying the most informative genes in the dataset

var = genes_train_ok.var(axis=0) # columns 

# Keeping the top 1000 genes 

top_1000_genes = var.sort_values(ascending=False).head(1000) 
top_1000_genes


## **Preprocessing**

Goals of this step:
* cleaning: NaN handling
* log transform (log1p)
* scaling (StandardScaler)

In [ ]:
# Reducing the dataset to 1000 genes only

genes_reduced = genes_train_ok[top_1000_genes.index]
genes_reduced.head 
genes_reduced.shape 

In [ ]:
# Checking if the data is already log transformed

np.min(genes_reduced.values), np.max(genes_reduced.values)

Considering the range of the values and the fact that there are both negative and positive values, the dataset is not log-transformed yet. 

Log-transformation of the data is not mandatory. For microarray datasets, it can be useless sometimes or redundant with the initial preprocessing. So the PCA and clustering will be applied on both the baseline dataset and the log-transformed dataset. There will be **two pipelines**.

The point of the log transformation is to correct the distribution of the data (skewness). It rebalances the influence of the extreme values. With the scaling, the variables become comparable between patients, making it easier for the clustering algorithm to compare distances between them. 

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
# Pipeline 1: No log-transformation of the data - Baseline

# StandardScaler

scaler = StandardScaler()
genes_reduced_scaled = scaler.fit_transform(genes_reduced)

In [ ]:
# Pipeline 2: log-transformation of the data 

# Shifting the data to make them all positive

genes_reduced_shifted = genes_reduced - genes_reduced.min().min() +1
# Here pd calculates the min per column => series => min of this series, min among the min = min of the whole df

# Log transform on positive values 

genes_reduced_log = np.log(genes_reduced_shifted)
scaler = StandardScaler()
genes_reduced_log_scaled = scaler.fit_transform(genes_reduced_log)

In [ ]:
# Checking the distribution of the data after P1 and P2

sns.kdeplot(genes_reduced_scaled.flatten(), label='scaled')
sns.kdeplot(genes_reduced_log_scaled.flatten(), label='log + scaled')
plt.legend()

The curves overlap, suggesting that shifting and log-transforming the data has no noticeable impact on the distribution of the data. Thus, the standardized data will be used for the rest of the analysis.

## **Dimensionality reduction**

Goals of this step: reduce dimensionality through PCA to make the clustering easier

38 patients, 1000 dimensions

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
# Checking that the shape corresponds to n_patients, n_genes

genes_reduced_scaled.shape 

In [ ]:
# PCA

pca = PCA()
genes_pca = pca.fit_transform(genes_reduced_scaled)

In [ ]:
# Explained variance: which components account for the majority of the variation between patients?
# Allows us to choose the number of components to keep for the clustering

plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.xlabel('Number of components')
plt.ylabel('Cumulated explained variance')

We see that 90% of the variance between patients is explained by the first 20 principal components. It suggests a strong redundancy between genes and justifies the use of dimensionality reduction before clustering. So only these 20 principal components will be kept for the clustering. 

In [ ]:
# Putting the patients as indexes in the cancer_type dataset to match the indexes of the gene 
# expression matrix

cancer_type_indexed = cancer_type.set_index('patient')
print(cancer_type_indexed.index[:10]) 

In [ ]:
# Visualization 

# 2D PCA
pca_viz = PCA(n_components=2) # creating the PCA and telling python we want to project the data in a 2D space
genes_viz = pca_viz.fit_transform(genes_reduced_scaled) # fitting the PCA to the gene set (the PCA looks for the PC1 and the PC2) and projecting each patient on the PC1 and PC2 axis

# Labels to color by cancer type
# Converting the indexes to float so that they can match those of the cancer_type_indexed dataset
genes_train_ok.index = genes_train_ok.index.astype(int)
y_train = cancer_type_indexed.loc[genes_train_ok.index, 'cancer'] # extracting the cancer label for each of the 38 patients (patient 38 included)

# Df for the plot 
pca_df = pd.DataFrame({
    'PC1': genes_viz[:,0], # every lines, PC1 
    'PC2': genes_viz[:,1], # every lines, PC2
    'Cancer': y_train # adding the AML and ALL label to every patient id to color the data points
})

# Plot
plt.figure(figsize=(8,6))

sns.scatterplot(
    data = pca_df,
    x = 'PC1', # horizontal axis = PC1
    y = 'PC2', # vertical axis = PC2
    hue = 'Cancer', # coloring based on the cancer type
    s = 100 # point size
) 

plt.xlabel(f"PC1({pca_viz.explained_variance_ratio_[0]*100:.1f}%)") # % of variance explained by PC1
plt.ylabel(f"PC2({pca_viz.explained_variance_ratio_[1]*100:.1f}%)") # % of variance explained by PC2
plt.title('PCA on the top 1000 most varying genes of the Golub and al. dataset - Colored by cancer type')
plt.axhline(0, linestyle='--', linewidth=0.5)
plt.axvline(0, linestyle='--', linewidth=0.5)

plt.tight_layout()
plt.show()

There is a clear distinction between ALL and AML patients on the PC1 axis which suggests that the main factor explaining the difference in gene expression patterns between patients is the leukemia type. 
The fact that the data is separated on the PC2 axis, which explains "only" 14% of the variance between patients, means that the cancer type is an important source of variation but not the main one. 

In [ ]:
y_train.head()

In [ ]:
cancer_type.columns

In [ ]:
print(cancer_type.head())
print(cancer_type.shape)

print(genes_reduced_scaled.shape)

In [ ]:
print(len(y_train))
print(genes_reduced_scaled.shape[0]) # number of rows only
# they are the same => perfect

## **Clustering**

Goals of this step: make gene clusters based on similarity, produce an understandable representation to visualize these clusters.

The clustering must be done on the first 20 principal components identified as the ones responsible for most of the variation between patients.

In [ ]:
# Creating the PCA clustering object

pca_clustering = PCA(n_components=20) # creating a PCA object that will contain the first 20 PCs
genes_cluster = pca_clustering.fit_transform(genes_reduced_scaled) # applying the PCA algorithm to the dataset

### K-means

In [ ]:
# Creating the kmeans clustering model

from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=42) # creating the kmeans object, asking the algorithm to separate the patients in 2 groups (because 2 cancer types)

clusters_kmeans = kmeans.fit_predict(genes_cluster) # fit_predict learns where to put the center of the clusters (fit) then links each patient to a cluster (predict)
# the kmeans algorithm tries to minimize the distance between the patients and the centroid of the cluster

In [ ]:
# Visualizing the clusters on the 2D PCA

pca_df['Cluster kmeans'] = clusters_kmeans # adding the clusters to the pca df

# Coloring the patients based on the cluster they were associated with
sns.scatterplot(
    data = pca_df,
    x = 'PC1',
    y = 'PC2',
    hue = 'Cluster kmeans',
    palette = 'deep',
    s = 100
)

# Comparing the 0/1 labels to the AML/ALL labels
pd.crosstab(clusters_kmeans, y_train) # creating a contingency table

Considering the previous PCA visualization, the k-means algorithms is quite performing at recognizing the two cancer types.

### Hierarchical clustering

This algorithm builds a hierarchy between patients based on their similarity. At first, each patient is a cluster. Then the clusters that are close to each other are merged into one.

In [ ]:
# Hierarchical clustering

from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

linked = linkage(genes_pca, method = 'ward')

In [ ]:
# Clusters extraction 

clusters_hier = fcluster(linked, t=2, criterion = 'maxclust')
# t=2: 2 final clusters
# maxclust: cut the tree to obtain 2 groups
print(clusters_hier)

In [ ]:
# Heatmap + dendrogram

sns.clustermap(
    genes_reduced_scaled,
    method = 'ward',
    metric = 'euclidean',
    cmap = 'viridis',
    figsize = (12,12)
)

This heatmap is not readable because there are too much genes so we will use only the top 50 genes to make it easier to read.

In [ ]:
top_50_genes = genes_reduced.var().sort_values(ascending = False).head(50).index

heatmap_data = genes_reduced[top_50_genes].copy()

# Adding annotations to clearly identify ALL and AML patients
patients_colors = y_train.map({
    'ALL' : 'blue',
    'AML' : 'red'
})

patients_colors.index = heatmap_data.index # forcing the indexes of both dataframes to align

# Generating the heatmap
sns.clustermap(
    heatmap_data,
    method = 'ward', 
    z_score = 1, # standardization of each gene (no need to rescale), z_score=1 standardizes by column
    cmap = 'viridis',
    row_colors = patients_colors,
    figsize = (12,12)
)

# Adding a legend to the heatmap
import matplotlib.patches as mpatches

blue_patch = mpatches.Patch(color = 'blue', label = 'ALL')
red_patch = mpatches.Patch(color = 'red', label = 'AML')

plt.legend(handles = [blue_patch, red_patch],
          bbox_to_anchor = (1,1))

The heatmap, along with the hierarchical clustering, highlights gene expression patterns that are specific to each type of leukemia. This could be explained by subtype-specific gene expression or signaling pathways. Interstingly, a few differences in the transcriptomic profile are noticed inside the ALL group and the AML group, suggesting subtype heterogeneity. For instance, it could be linked to variations in molecular signature or to the existence of different subtypes of each type of leukemia.

### Clusters validation

In [ ]:
# Comparing k-means clusters to AML/ALL labels

pd.crosstab(clusters_kmeans, y_train)

In [ ]:
# Comparing hierarchical clusters to AML/ALL labels

pd.crosstab(clusters_hier, y_train)

In [ ]:
# Visualizing the hierarchical clusters on the pca 

pca_df['HierCluster'] = clusters_hier

plt.figure(figsize=(8,6))

sns.scatterplot(
    data = pca_df,
    x = 'PC1',
    y = 'PC2', 
    hue = 'HierCluster',
    s = 100
)

plt.title('Hierarchical clustering on PCA space')
plt.show()

The hierarchical clustering algorithm performs worst at finding AML/ALL clusters than the k-means algorithm. Two ALL patients are isolated from the others. They could have a very similar transcriptomic profile, which would explain the fact that they are isolated as independent clusters. 
So different clustering algorithms can identify different structures of the same data. K-means distinguish AML and ALL patients better whereas the hierarchical clustering with the ward linkage method isolates an atypical subset of patients from the others. 

Now other linkage methods and cluster numbers will be tested to optimize the clustering alogorithms. 

#### Test: 'complete' linkage method - hierarchical clustering

In [ ]:
# Testing other linkage methods of hierarchical clustering 
# Method: complete

# Making the heatmap
sns.clustermap(
    heatmap_data,
    method = 'complete', 
    z_score = 1,
    cmap = 'viridis',
    row_colors = patients_colors,
    figsize = (12,12)
)

# Adding a legend to the heatmap
import matplotlib.patches as mpatches

blue_patch = mpatches.Patch(color = 'blue', label = 'ALL')
red_patch = mpatches.Patch(color = 'red', label = 'AML')

plt.legend(handles = [blue_patch, red_patch],
          bbox_to_anchor = (1,1))

# Clusters extraction 
clusters_hier_1 = fcluster(linked, t=2, criterion = 'maxclust')
print(clusters_hier_1)

In [ ]:
# Visualizing the hierarchical clusters on the pca 

pca_df['HierCluster_1'] = clusters_hier_1

plt.figure(figsize=(8,6))

sns.scatterplot(
    data = pca_df,
    x = 'PC1',
    y = 'PC2', 
    hue = 'HierCluster_1',
    s = 100
)

plt.title('Hierarchical clustering on PCA space')
plt.show()

#### Test: 'average' linkage method - hier. clustering

In [ ]:
# Testing other linkage methods of hierarchical clustering 
# Method: average

# Making the heatmap
sns.clustermap(
    heatmap_data,
    method = 'average', 
    z_score = 1, 
    cmap = 'viridis',
    row_colors = patients_colors,
    figsize = (12,12)
)

# Adding a legend to the heatmap
import matplotlib.patches as mpatches

blue_patch = mpatches.Patch(color = 'blue', label = 'ALL')
red_patch = mpatches.Patch(color = 'red', label = 'AML')

plt.legend(handles = [blue_patch, red_patch],
          bbox_to_anchor = (1,1))

# Clusters extraction 
clusters_hier_2 = fcluster(linked, t=2, criterion = 'maxclust')
print(clusters_hier_2)

In [ ]:
# Visualizing the hierarchical clusters on the pca 

pca_df['HierCluster_2'] = clusters_hier_2

plt.figure(figsize=(8,6))

sns.scatterplot(
    data = pca_df,
    x = 'PC1',
    y = 'PC2', 
    hue = 'HierCluster_2',
    s = 100
)

plt.title('Hierarchical clustering on PCA space')
plt.show()

#### Test: number of clusters t=3

In [ ]:
# Clusters extraction 
clusters_hier_3 = fcluster(linked, t=3, criterion = 'maxclust')
print(clusters_hier_3)

# Visualizing the hierarchical clusters on the pca 
pca_df['HierCluster_3'] = clusters_hier_3

plt.figure(figsize=(8,6))

sns.scatterplot(
    data = pca_df,
    x = 'PC1',
    y = 'PC2', 
    hue = 'HierCluster_3',
    s = 100
)

plt.title('Hierarchical clustering on PCA space')
plt.show()

#### Test: number of clusters t=4

In [ ]:
# Clusters extraction 
clusters_hier_4 = fcluster(linked, t=4, criterion = 'maxclust')
print(clusters_hier_4)

# Visualizing the hierarchical clusters on the pca 
pca_df['HierCluster_4'] = clusters_hier_4

plt.figure(figsize=(8,6))

sns.scatterplot(
    data = pca_df,
    x = 'PC1',
    y = 'PC2', 
    hue = 'HierCluster_4',
    s = 100
)

plt.title('Hierarchical clustering on PCA space')
plt.show()

#### Conclusions on the tests

Testing different linkage method slightly modifies the hierarchical structure of the dendrogram but consistently isolates two atypical ALL patients as a separate cluster. This suggests that these patients have transcriptomic profiles that are very distinct from the rest of the cohort and their isolation is robust to the choice of linkage method.

However, increasing the number of clusters from 2 to 3 or 4 reveals additional subgroups within both AML and ALL patients (particularly AML patients), highlighting intra-group heterogeneity. With t=4, the hierarchical clustering structure becomes closer to the one identified by the k-means algorithm, which more clearly separates AML and ALL patients. Interestingly, the two atypical ALL patients remain isolated even with a higher number of clusters suggesting the presence of a particularly distinct transcriptomic subgroup.

#### ARI and silhouette score 

ARI: to what extent does the clustering correspond to the actual ALL/AML labels?

In [ ]:
# Adjusted Rand Index (ARI)

from sklearn.metrics import adjusted_rand_score

print(adjusted_rand_score(y_train, clusters_kmeans)) # k-means algorithm
print(adjusted_rand_score(y_train, clusters_hier))   # 'ward' linkage method
print(adjusted_rand_score(y_train, clusters_hier_1)) # 'complete' linkage method
print(adjusted_rand_score(y_train, clusters_hier_2)) # 'average' linkage method
print(adjusted_rand_score(y_train, clusters_hier_3)) # t=3
print(adjusted_rand_score(y_train, clusters_hier_4)) # t=4

The k-means algorithm is the one that performs best at identifying ALL and AML patients. 
The hierarchical clustering performs poorly at separating ALL vs AML patients. However, increasing the number of clusters from 2 to 3 improves the algorithm's performance which better captures the structure of the data. Allowing for 4 clusters causes the ARI to drop so we partially lose the ALL/AML structuring of the data.

Silhouette score: evaluation of the compaction and of the separation of the clusters

In [ ]:
# Silhouette score 

from sklearn.metrics import silhouette_score

print(silhouette_score(genes_pca, clusters_kmeans))
print(silhouette_score(genes_pca, clusters_hier))
print(silhouette_score(genes_pca, clusters_hier_1))
print(silhouette_score(genes_pca, clusters_hier_2))
print(silhouette_score(genes_pca, clusters_hier_3))
print(silhouette_score(genes_pca, clusters_hier_4))

Surprisingly, although the k-means algorithms performs well at identifying leukemia types, the clusters it creates are of poor geometrical quality (not compact). 
The hierarchical clustering with t=2 allows for better geometrical quality of the clusters (more compact), even though it performs poorly at separating ALL from AML patients. 
This result suggests that the leukemia type is not the only factor that influences/determines the structure of the data. 

## **Classification model**

Goals of this step: establish a classification model to predict patients cancer type

Here, we have a binary classification problem. Thus, three models would be adequate: logistic regression (easily interpretable), SVM (performing on datasets with only a few observations), and a random forest (flexible, can model complex relationships between variables, risk of overfitting considering the low number of observations in this dataset). The three of them are going to be tested to asses which one is the best at predicting patients cancer type.

### SVM

#### Training the model

In [ ]:
# Implementing the model 

from sklearn.svm import SVC

svm = SVC(
    kernel='linear', # the most simple kernel, good for baseline modeling
    random_state=42 # making sure that the model produces the same results at each run 
)

# Training the model 

X_train = genes_cluster # the data is normalized, scaled and the pca algo has been applied to it
y_train = cancer_type_indexed.loc[genes_train_ok.index, 'cancer']

svm.fit(genes_cluster, y_train)

#### Preprocessing of the validation dataset

In [ ]:
genes_independent.head(5)

In [ ]:
genes_independent.shape

In [ ]:
# The test dataset must undergo the exact same transformations applied on the train dataset

X_test = genes_independent

# Dropping the 'call' columns

cols_drop = [col for col in X_test.columns if 'call' in col.lower()]
X_test_ok = X_test.drop(X_test[cols_drop], axis=1)
X_test_ok.head()
X_test_ok.shape # 36 patients 

# Transposing the columns and the rows 

X_test_t = X_test_ok.transpose()

In [ ]:
# Converting the data to numerical values 

print(X_test_t.dtypes) 

# Putting genes info in another dataset

X_test_info = X_test_t.iloc[0:2,:]
X_test_t = X_test_t.drop(['Gene Description', 'Gene Accession Number'], axis=0)

# Converting the numerical dataset to float

X_test_t = X_test_t.apply(pd.to_numeric, errors = 'coerce')

# Checking that no NaN after the conversion

X_test_t.isna().mean().mean()

In [ ]:
X_test_info.head()

In [ ]:
# Filtering - 1000 most informative genes 

X_test_1000 = X_test_t[top_1000_genes.index]

In [ ]:
# Scaling of the data 

X_test_scaled = scaler.transform(X_test_1000)

In [ ]:
# PCA on the test dataset

X_test_pca = pca_clustering.transform(X_test_scaled)
X_test_pca.shape

#### Evaluation of the model on the validation dataset

In [ ]:
# Making predictions

preds_svm = svm.predict(X_test_pca)

To evaluate the model, we will use the accuracy score, a confusion matrix and the F1 score. 

In [ ]:
# Importing the different metrics algorithm 

from sklearn.metrics import (
accuracy_score, # important
confusion_matrix,# important
ConfusionMatrixDisplay,
precision_score,
recall_score,
f1_score, # important (compromise between precision and recall)
classification_report # summarizes all the scores
)

In [ ]:
cancer_type_indexed.head()

In [ ]:
# Creating the target labels 

X_test_t.index = X_test_t.index.astype(int)
y_test = cancer_type_indexed.loc[X_test_t.index, 'cancer']

In [ ]:
y_test.unique()

In [ ]:
# Performance metrics 

print('Accuracy:', accuracy_score(y_test, preds_svm))

print('\nConfusion matrix:')
print(confusion_matrix(y_test, preds_svm))

print('\nF1 AML:', f1_score(y_test, preds_svm, pos_label='AML')) # arbitrary choosing to establish
                                                             # the AML class as the positive label

print('\nClassification report:')
print(classification_report(y_test, preds_svm))

In [ ]:
# Confusion matrix display for the SVM model 

ConfusionMatrixDisplay.from_predictions(
    y_test,
    preds_svm
)
plt.show()

The model perfectly identifies AML patients but occasionnally misclassifies ALL patients as AML. Therefore, it performs better on the AML class than on the ALL. 

#### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression 

In [ ]:
logreg = LogisticRegression(
    max_iter=5000,
    random_state=42
)

logreg.fit(X_train, y_train)

preds_lr = logreg.predict(X_test_pca)

print(classification_report(y_test, preds_lr))

In [ ]:
# Confusion matrix display for the logreg model 

ConfusionMatrixDisplay.from_predictions(
    y_test, 
    preds_lr
)
plt.show()

The same observation goes for the logistic regression model. This model misclassifies more ALL patients than the SVM, which is consistent with its lower prediction accuracy (82% vs 88%). 

#### Random Forest

In [ ]:
# Importing the RF algorithm

from sklearn.ensemble import RandomForestClassifier

In [ ]:
rf1 = RandomForestClassifier(random_state=42, n_estimators=200)

rf1.fit(X_train, y_train)

preds_rf1 = rf1.predict(X_test_pca)

print(classification_report(y_test, preds_rf1))

In [ ]:
rf2 = RandomForestClassifier(random_state=42, n_estimators=100)

rf2.fit(X_train, y_train)

preds_rf2 = rf2.predict(X_test_pca)

print(classification_report(y_test, preds_rf2))

In [ ]:
rf3 = RandomForestClassifier(random_state=42, n_estimators=150)

rf3.fit(X_train, y_train)

preds_rf3 = rf3.predict(X_test_pca)

print(classification_report(y_test, preds_rf3))

The random forest model performs poorly at predicting patients cancer type with an accuracy that does not exceed 47%. This is the worst performing model of the three tested in our case. 

#### Interpretation of the models

The SVM classifier outperformed logistric regression, achieving an accuracy of 88% compared with 82%. Both models showed a higher sensitivity to ALL samples than AML samples, suggesting that AML profiles may be more heterogenous and therefore harder to classify. It is coherent with the heterogeneity observed in the AML population in the clustering. The random forest model achieved substantially lower performance than both SVM and logistic regression. This may be explained by the small sample size available for training (38 patients) and by the fact that the AML/ALL separation appears to be linear, as suggested by the PCA visualization. In this context, linear classifiers are better suited to capture the underlying structure of the data than tree-based ensemble models. 